# Location-specific carbon price comparison

Green methanol and green ammonia with 100% renewable electricity input at three locations: Escondida, Intersection, and Puerto Coloso.

In [1]:
import pandas as pd

# ==========================================================
# Location-specific carbon price comparison for green methanol
# and green ammonia with 100% renewable electricity input.
# ==========================================================

# --------------------------
# General assumptions
# --------------------------
exchange_rate_dkk_to_eur = 0.1338

# Keep the 2050 allowance from the previous case for reference.
# Not used in the carbon price formula below, but kept in case you want it later.
eua_2050_dkk = 4633.0
eua_2050_eur = eua_2050_dkk * exchange_rate_dkk_to_eur

# Fossil benchmark prices (same as previous scenarios / assumptions used earlier)
fossil_methanol_price_eur_per_t = 350.0
fossil_ammonia_price_eur_per_t = 400.0

# Industrial emission benchmarks (same as previous scenarios / assumptions used earlier)
# Methanol: t CO2 per t MeOH
methanol_industrial_emissions_low = 2.05
methanol_industrial_emissions_high = 2.863

# Ammonia: t CO2 per t NH3
ammonia_industrial_emissions_low = 1.9
ammonia_industrial_emissions_high = 2.6

# Product energy contents used to convert LCOE [EUR/MWh] to [EUR/t]
# Methanol LHV ~ 19.7 GJ/t -> 5.4722 MWh/t
methanol_energy_density_j_per_t = 19.7e9
methanol_energy_density_mwh_per_t = methanol_energy_density_j_per_t / 3.6e9

# Ammonia LHV is approximately 18.6 GJ/t -> 5.1667 MWh/t
# If you use another basis (e.g. HHV), change this constant.
ammonia_energy_density_j_per_t = 18.6e9
ammonia_energy_density_mwh_per_t = ammonia_energy_density_j_per_t / 3.6e9

# 100% renewable electricity input -> no operational emissions from grid
renewable_grid_emissions_kg_co2_per_wh = 0.0

# Base-case grid emissions for the Escondida scenario (methanol and ammonia)
# 152 g CO2/kWh = 0.152 t CO2/MWh
escondida_grid_emissions_g_co2_per_kwh = 152.0
escondida_grid_emissions_t_co2_per_mwh = escondida_grid_emissions_g_co2_per_kwh / 1000.0


In [2]:
# --------------------------
# Location-specific LCOE data [EUR/MWh]
# --------------------------
# Columns: scenario, At Minera Escondida / Local Demand, At Puerto Coloso / Export
methanol_lcoe = {
    'Escondida': {
        'scenario_lcoe': 79.07,
        'mine_local': 79.07,
        'port_export': 80.42,
    },
    'Intersection': {
        'scenario_lcoe': 78.31,
        'mine_local': 79.59,
        'port_export': 79.14,
    },
    'Puerto Coloso': {
        'scenario_lcoe': 79.39,
        'mine_local': 82.78,
        'port_export': 79.39,
    },
}

ammonia_lcoe = {
    'Escondida': {
        'scenario_lcoe': 126.02,
        'mine_local': 126.02,
        'port_export': 126.69,
    },
    'Intersection': {
        'scenario_lcoe': 128.77,
        'mine_local': 129.03,
        'port_export': 129.19,
    },
    'Puerto Coloso': {
        'scenario_lcoe': 129.60,
        'mine_local': 130.29,
        'port_export': 129.60,
    },
}

In [3]:
# --------------------------
# Helpers
# --------------------------
def lcoe_to_lcot(lcoe_eur_per_mwh: float, mwh_per_t: float) -> float:
    """Convert LCOE [EUR/MWh] to cost per tonne product [EUR/t]."""
    return lcoe_eur_per_mwh * mwh_per_t


def electricity_emissions_tco2_per_t(scenario_name: str, mwh_per_t: float) -> float:
    """Electricity-related product emissions [tCO2/t]. Escondida uses 152 gCO2/kWh."""
    if scenario_name == 'Escondida':
        return mwh_per_t * escondida_grid_emissions_t_co2_per_mwh
    return 0.0


def carbon_price_eur_per_tco2(
    product_cost_eur_per_t: float,
    fossil_cost_eur_per_t: float,
    fossil_emissions_tco2_per_t: float,
    product_emissions_tco2_per_t: float = 0.0,
) -> float:
    """Compute necessary carbon price [EUR/tCO2]."""
    denom = fossil_emissions_tco2_per_t - product_emissions_tco2_per_t
    if denom <= 0:
        return float('nan')
    return (product_cost_eur_per_t - fossil_cost_eur_per_t) / denom


def build_rows(
    fuel_name: str,
    lcoe_table: dict,
    mwh_per_t: float,
    fossil_cost_eur_per_t: float,
    industrial_low: float,
    industrial_high: float,
) -> list[dict]:
    rows = []
    for scenario_name, values in lcoe_table.items():
        scenario_lcoe = values['scenario_lcoe']
        for basis_key, basis_label in [
            ('mine_local', 'At Minera Escondida / Local Demand'),
            ('port_export', 'At Puerto Coloso / Export'),
        ]:
            lcoe_eur_per_mwh = values[basis_key]
            lcoe_eur_per_t = lcoe_to_lcot(lcoe_eur_per_mwh, mwh_per_t)

            product_emissions = electricity_emissions_tco2_per_t(
                scenario_name=scenario_name,
                mwh_per_t=mwh_per_t,
            )

            cp_low = carbon_price_eur_per_tco2(
                product_cost_eur_per_t=lcoe_eur_per_t,
                fossil_cost_eur_per_t=fossil_cost_eur_per_t,
                fossil_emissions_tco2_per_t=industrial_low,
                product_emissions_tco2_per_t=product_emissions,
            )
            cp_high = carbon_price_eur_per_tco2(
                product_cost_eur_per_t=lcoe_eur_per_t,
                fossil_cost_eur_per_t=fossil_cost_eur_per_t,
                fossil_emissions_tco2_per_t=industrial_high,
                product_emissions_tco2_per_t=product_emissions,
            )

            rows.append({
                'Fuel': fuel_name,
                'Scenario': scenario_name,
                'Basis': basis_label,
                'LCOE [EUR/MWh]': lcoe_eur_per_mwh,
                'LCOE [EUR/t]': lcoe_eur_per_t,
                'Fossil benchmark [EUR/t]': fossil_cost_eur_per_t,
                'Industrial emissions low [tCO2/t]': industrial_low,
                'Industrial emissions high [tCO2/t]': industrial_high,
                'Product emissions [tCO2/t]': product_emissions,
                'Carbon price low [EUR/tCO2]': cp_low,
                'Carbon price high [EUR/tCO2]': cp_high,
                'Scenario LCOE [EUR/MWh]': scenario_lcoe,
            })
    return rows

In [4]:
# --------------------------
# Build results
# --------------------------
rows = []
rows += build_rows(
    fuel_name='Methanol',
    lcoe_table=methanol_lcoe,
    mwh_per_t=methanol_energy_density_mwh_per_t,
    fossil_cost_eur_per_t=fossil_methanol_price_eur_per_t,
    industrial_low=methanol_industrial_emissions_low,
    industrial_high=methanol_industrial_emissions_high,
)
rows += build_rows(
    fuel_name='Ammonia',
    lcoe_table=ammonia_lcoe,
    mwh_per_t=ammonia_energy_density_mwh_per_t,
    fossil_cost_eur_per_t=fossil_ammonia_price_eur_per_t,
    industrial_low=ammonia_industrial_emissions_low,
    industrial_high=ammonia_industrial_emissions_high,
)

result = pd.DataFrame(rows)

# Nice ordering
result = result[
    [
        'Fuel', 'Scenario', 'Basis',
        'Scenario LCOE [EUR/MWh]', 'LCOE [EUR/MWh]', 'LCOE [EUR/t]',
        'Fossil benchmark [EUR/t]',
        'Industrial emissions low [tCO2/t]', 'Industrial emissions high [tCO2/t]',
        'Product emissions [tCO2/t]',
        'Carbon price low [EUR/tCO2]', 'Carbon price high [EUR/tCO2]',
    ]
]

# --------------------------
# Output
# --------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

print('\n=== Location-specific carbon price comparison ===')
print(result.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

# Optional: round for a compact table version
result_rounded = result.copy()
for col in result_rounded.columns:
    if pd.api.types.is_numeric_dtype(result_rounded[col]):
        result_rounded[col] = result_rounded[col].round(2)

# Save to Excel if desired
output_file_name = 'co2_prices_locations.xlsx'
try:
    with pd.ExcelWriter(output_file_name, engine='openpyxl') as writer:
        result.to_excel(writer, sheet_name='Results', index=False)
    print(f'\nSaved to {output_file_name}')
except Exception as exc:
    print(f'\nCould not save Excel file: {exc}')

# If running interactively, show the DataFrame too
try:
    display(result_rounded)
except Exception:
    print('\nRounded summary:')
    print(result_rounded.to_string(index=False))


=== Location-specific carbon price comparison ===
    Fuel      Scenario                              Basis  Scenario LCOE [EUR/MWh]  LCOE [EUR/MWh]  LCOE [EUR/t]  Fossil benchmark [EUR/t]  Industrial emissions low [tCO2/t]  Industrial emissions high [tCO2/t]  Product emissions [tCO2/t]  Carbon price low [EUR/tCO2]  Carbon price high [EUR/tCO2]
Methanol     Escondida At Minera Escondida / Local Demand                    79.07           79.07        432.69                    350.00                               2.05                                2.86                        0.83                        67.88                         40.71
Methanol     Escondida          At Puerto Coloso / Export                    79.07           80.42        440.08                    350.00                               2.05                                2.86                        0.83                        73.94                         44.35
Methanol  Intersection At Minera Escondida / Local Demand 

,Fuel,Scenario,Basis,Scenario LCOE [EUR/MWh],LCOE [EUR/MWh],LCOE [EUR/t],Fossil benchmark [EUR/t],Industrial emissions low [tCO2/t],Industrial emissions high [tCO2/t],Product emissions [tCO2/t],Carbon price low [EUR/tCO2],Carbon price high [EUR/tCO2]
0,Methanol,Escondida,At Minera Escondida / Local Demand,79.07,79.07,432.69,350.0,2.05,2.86,0.83,67.88,40.71
1,Methanol,Escondida,At Puerto Coloso / Export,79.07,80.42,440.08,350.0,2.05,2.86,0.83,73.94,44.35
2,Methanol,Intersection,At Minera Escondida / Local Demand,78.31,79.59,435.53,350.0,2.05,2.86,0.00,41.72,29.88
3,Methanol,Intersection,At Puerto Coloso / Export,78.31,79.14,433.07,350.0,2.05,2.86,0.00,40.52,29.02
4,Methanol,Puerto Coloso,At Minera Escondida / Local Demand,79.39,82.78,452.99,350.0,2.05,2.86,0.00,50.24,35.97
5,Methanol,Puerto Coloso,At Puerto Coloso / Export,79.39,79.39,434.44,350.0,2.05,2.86,0.00,41.19,29.49
6,Ammonia,Escondida,At Minera Escondida / Local Demand,126.02,126.02,651.10,400.0,1.90,2.60,0.79,225.27,138.37
7,Ammonia,Escondida,At Puerto Coloso / Export,126.02,126.69,654.57,400.0,1.90,2.60,0.79,228.38,140.28
8,Ammonia,Intersection,At Minera Escondida / Local Demand,128.77,129.03,666.66,400.0,1.90,2.60,0.00,140.34,102.56
9,Ammonia,Intersection,At Puerto Coloso / Export,128.77,129.19,667.48,400.0,1.90,2.60,0.00,140.78,102.88
